In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import json
# from datasets import load_dataset

In [2]:
data = pd.read_parquet('../data.parquet')
# data = data[
#     (data.concept_genre == 'text') 
# ]
len(data)

1152216

In [3]:
a = data["output_concept"].dropna().unique().tolist()

with open("llama_output_concept.json", "w") as f:
    json.dump(a, f, ensure_ascii=False, indent=2)

In [4]:
prompts, rejected, accepted, steering_concepts = [], [], [], []

In [5]:
for input_prompt, input_group in tqdm(data.groupby('input'), total=len(data.input.unique())):
    negative = input_group[input_group.category == 'negative']
    if len(negative) > 0:
        for output_concept, output_group in input_group.groupby('output_concept'):
            positive = output_group[output_group.category == 'positive']
            if len(positive) > 0:
                temp_prompt = 'question:\n' + input_prompt + '\n' + 'steering concept:\n' + output_concept
                temp_accepted = positive.iloc[0]['output']
                temp_rejected = input_group[input_group.category == 'negative'].sample(n=1, random_state=44)['output'].iloc[0]
                # print(type(temp_rejected))
                if len(temp_prompt)<10:continue
                if len(temp_accepted)<10:continue
                if len(temp_rejected)<10:continue
                prompts.append(temp_prompt)
                rejected.append(temp_rejected)
                accepted.append(temp_accepted)
                steering_concepts.append(output_concept)

print(len(prompts), len(rejected), len(accepted), len(steering_concepts))

100%|██████████| 2995/2995 [00:20<00:00, 144.03it/s]

82586 82586 82586 82586


In [6]:
print(prompts[0])
print('__')
print(rejected[0])
print('__')
print(accepted[0])
print('__')



question:
40% of the mosquitos in Jack's area are infected with malaria. 20% of the mosquitos are infected with Zika virus. Without a vaccine, the chances of getting infected with either virus after getting bitten by an infected mosquito are 50%. Jack is taking an experimental malaria vaccine that reduces the chances of getting infected after getting bitten by 50%. If Jack gets bitten by a random mosquito, what is the percentage chance he catches either Zika virus or malaria?
steering concept:
complex mathematical statements and arguments
__
To find the percentage chance Jack catches either Zika virus or malaria, we need to consider the following scenarios:

1. Jack gets bitten by a mosquito infected with malaria but not Zika virus.
2. Jack gets bitten by a mosquito infected with Zika virus but not malaria.
3. Jack gets bitten by a mosquito infected with both malaria and Zika virus.

The probability of getting bitten by a mosquito infected with malaria but not Zika virus is 40% (malari

In [9]:
# Split by steering concept only: each unique output_concept appears in exactly one of train/valid/test (75/10/15).
SPLIT_RANDOM_SEED = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.75, 0.10, 0.15

unique_concepts = list(dict.fromkeys(steering_concepts))
rng = np.random.default_rng(SPLIT_RANDOM_SEED)
order = rng.permutation(len(unique_concepts))
shuffled = [unique_concepts[i] for i in order]

n_concepts = len(shuffled)
n_train = int(TRAIN_FRAC * n_concepts)
n_val = int(VAL_FRAC * n_concepts)
# Assign any remainder from rounding to test so all concepts are covered once
n_test = n_concepts - n_train - n_val

train_concepts = set(shuffled[:n_train])
val_concepts = set(shuffled[n_train : n_train + n_val])
test_concepts = set(shuffled[n_train + n_val :])

assert train_concepts.isdisjoint(val_concepts)
assert train_concepts.isdisjoint(test_concepts)
assert val_concepts.isdisjoint(test_concepts)
assert train_concepts | val_concepts | test_concepts == set(unique_concepts)

def make_record(p, c, r):
    return {
        "prompt": [{"role": "user", "content": p}],
        "chosen": [{"role": "assistant", "content": c}],
        "rejected": [{"role": "assistant", "content": r}],
    }

records_train, records_val, records_test = [], [], []
for p, c, r, concept in zip(prompts, accepted, rejected, steering_concepts):
    rec = make_record(p, c, r)
    if concept in train_concepts:
        records_train.append(rec)
    elif concept in val_concepts:
        records_val.append(rec)
    else:
        records_test.append(rec)

print(
    f"Concepts: train={len(train_concepts):,} valid={len(val_concepts):,} test={len(test_concepts):,} "
    f"(total unique={n_concepts:,})"
)
print(
    f"Examples: train={len(records_train):,} valid={len(records_val):,} test={len(records_test):,}"
)

for split_name, recs, fname in [
    ("train", records_train, "v2_train.jsonl"),
    ("valid", records_val, "v2_valid.jsonl"),
    ("test", records_test, "v2_test.jsonl"),
]:
    with open(fname, "w", encoding="utf-8") as f:
        for ex in recs:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"Wrote {len(recs):,} examples to {fname}")


Concepts: train=11,779 valid=1,570 test=2,357 (total unique=15,706)
Examples: train=61,857 valid=8,282 test=12,447
Wrote 61,857 examples to v2_train.jsonl
Wrote 8,282 examples to v2_valid.jsonl
Wrote 12,447 examples to v2_test.jsonl


In [10]:
# Quick sanity-check: inspect one train example (index 10k if available)
idx = 10_000 if len(records_train) > 10_000 else 0
print(records_train[idx] if records_train else "no train records")


{'prompt': [{'role': 'user', 'content': 'question:\nHow often are the Olympics held?\nsteering concept:\nreferences to specific individuals or entities, particularly highlighting prominent names in discussions'}], 'chosen': [{'role': 'assistant', 'content': 'The Summer and Winter Olympics occur every four years, a structure famously endorsed by figures like Pierre de Coubertin, who played a crucial role in reviving the modern Olympic Games, and prominent athletes like Usain Bolt and Michael Phelps further emphasize the global significance of these events.'}], 'rejected': [{'role': 'assistant', 'content': "The Olympics are held every 4 years. There are two main types of Olympic Games: \n\n1. Summer Olympics: These are held every 4 years, usually in July or August. The first Summer Olympics were held in 1896 in Athens, Greece.\n\n2. Winter Olympics: These are also held every 4 years, but 2 years after the Summer Olympics. The first Winter Olympics were held in 1924 in Chamonix, France.\n